In [1]:
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BitsAndBytesConfig
os.environ["DISABLE_TQDM"] = "1"

Load Documents

In [2]:
DATASET_PATH = "../Dataset"

def load_documents():
    docs = {}
    doc_dir = os.path.join(DATASET_PATH, "documents")
    
    for file in os.listdir(doc_dir):
        if file.endswith(".txt"):
            name = file.replace(".txt", "")
            file_path = os.path.join(doc_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                docs[name] = f.read()
    return docs

def load_all_qa():
    all_qa = []
    qa_dir = os.path.join(DATASET_PATH, "qa")
    
    for file in os.listdir(qa_dir):
        if file.endswith(".json"):
            file_path = os.path.join(qa_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                all_qa.extend(json.load(f))
    return all_qa

Load LLM models

In [3]:
# def load_model(model_name):
#     tokenizer = AutoTokenizer.from_pretrained(model_name)

#     quant_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,
#         llm_int8_enable_fp32_cpu_offload=True
#     )

#     model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         device_map="auto",
#         quantization_config=quant_config,
#         offload_folder="offload",
#         offload_state_dict=True,
#         low_cpu_mem_usage=True
#     )

#     model.eval()
#     return tokenizer, model


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",              
        quantization_config=quant_config,
        offload_folder="offload",       
        low_cpu_mem_usage=True
    )

    model.eval()
    return tokenizer, model

In [4]:
#MISTRAL = "mistralai/Mistral-7B-Instruct-v0.3"

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"

Metric Utils

In [5]:
# Token Counter
def count_tokens(tokenizer, text):
    return len(tokenizer(text)["input_ids"])

In [6]:
# Exact Match
def exact_match(pred, gold):
    return int(pred.strip() == gold.strip())

In [7]:
# def generate(model, tokenizer, prompt):

#     # Full Pipeline Timer
#     start_total = time.time()

#     # Count tokens BEFORE truncation
#     full_prompt_tokens = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])

#     # Tokenization with truncation for the model
#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length= tokenizer.model_max_length
#     )

#     inputs = {k: v.to(model.device) for k, v in inputs.items()}

#     prompt_tokens_used = inputs["input_ids"].shape[1]

#     # Model generation
#     output = model.generate(**inputs, max_new_tokens=100)

#     end_total = time.time()

#     # Decode output
#     text = tokenizer.decode(output[0], skip_special_tokens=True)

#     output_tokens = output.shape[-1]

#     # TRUE END-TO-END LATENCY
#     latency = end_total - start_total

#     return text, latency, full_prompt_tokens, prompt_tokens_used, output_tokens

def generate(model, tokenizer, prompt):
    torch.cuda.empty_cache()

    start = time.time()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096   
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,   
            temperature=0.0,
            do_sample=False
        )

    end = time.time()

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "text": text,
        "latency": end - start,
        "prompt_tokens": inputs["input_ids"].shape[1],
        "output_tokens": outputs.shape[1] - inputs["input_ids"].shape[1]
    }

In [9]:
# # Timed Generation
# def generate(model, tokenizer, prompt):
#     torch.cuda.empty_cache()
#     start = time.time()
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    
#     device = next(model.parameters()).device
#     inputs = {k: v.to(device) for k, v in inputs.items()}

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             temperature=0.0,    
#             do_sample=False
#         )
#     end = time.time()

#     output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

#     latency = end - start
#     input_tokens = inputs["input_ids"].shape[1]
#     output_tokens = outputs.shape[1] - input_tokens
#     torch.cuda.empty_cache()
#     return output_text, latency, input_tokens + output_tokens

LLM Systems

Prompt Only

In [8]:
def run_prompt_only(model, tokenizer, docs, qa_data):
    results = []

    answers = []

    for item in qa_data:
        doc_text = docs[item["document"]]

        prompt = f"""
Extract the exact answer span from the document.

Document:
{doc_text}

Question:
{item["question"]}

Answer:
"""

        gen_data = generate(model, tokenizer, prompt)
        prediction = gen_data["text"]

        results.append({
            "qa_id": item["id"],
            "system": "Prompt-Only",
            "latency": gen_data["latency"],
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"])
        })

        answers.append({
            "qa_id": item["id"],
            "question": item["question"],
            "document": item["document"],
            "prediction": prediction,
            "gold_answer": item["answer_text"]
        })
        torch.cuda.empty_cache() 

    with open("answers_prompt.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

RAG

In [9]:
def chunk_text(tokenizer, text, size=256, overlap=64):
    # Add truncation=False to ensure we get ALL tokens for the index
    inputs = tokenizer(text, truncation=False, add_special_tokens=False)
    tokens = inputs["input_ids"]
    chunks = []

    for i in range(0, len(tokens), size - overlap):
        chunk_tokens = tokens[i:i+size]
        chunks.append(tokenizer.decode(chunk_tokens))

    return chunks

In [10]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def build_index(docs, tokenizer, embedder):
    all_chunks = []
    
    for doc_id, text in docs.items():
        # Break the 150K document into manageable pieces
        chunks = chunk_text(tokenizer, text)
        all_chunks.extend(chunks)

    # Encode in batches to prevent CPU memory spikes
    # SentenceTransformers .encode() has a 'batch_size' parameter
    embeddings = embedder.encode(all_chunks, batch_size=8, show_progress_bar=True)
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    for i in range(0, len(embeddings), 500):
        batch = embeddings[i:i+500]
        index.add(np.array(batch).astype('float32'))
    
    return index, all_chunks

In [11]:
def run_rag(model, tokenizer, index, chunks, qa_data):
    results = []

    answers = []

    for item in qa_data:
        q_emb = embedder.encode(item["question"], convert_to_numpy=True).reshape(1, -1)

        start_retrieval = time.time()
        D, I = index.search(q_emb, 5)
        retrieval_time = time.time() - start_retrieval

        retrieved_chunks = [chunks[i] for i in I[0]]
        context = context = "\n\n".join(retrieved_chunks[:3])
        if len(context) > 4000:
            context = context[:4000]

        prompt = f"""
Extract exact answer span.

Context:
{context}

Question:
{item["question"]}

Answer:
"""

        gen_data = generate(model, tokenizer, prompt)
        
        prediction = gen_data["text"]

        recall = int(any(item["answer_text"] in c for c in retrieved_chunks))

        results.append({
            "qa_id": item["id"],
            "system": "RAG",
            "latency": gen_data["latency"], # You can add retrieval_time here if you want total latency
            "prompt_tokens_full": gen_data["prompt_tokens"],
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"]),
            "Recall": recall
        })

        answers.append({
            "qa_id": item["id"],
            "question": item["question"],
            "prediction": prediction,
            "gold_answer": item["answer_text"],
            "retrieved_chunks": retrieved_chunks
        })
        torch.cuda.empty_cache() 

    with open("answers_rag.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

Context Optimization

In [12]:
def run_context(docs, qa_data, model, tokenizer):
    results, answers = [], []

    def process(item):
        doc_text = docs.get(item.get("document", ""), "")
        query = str(item.get("article", "")).lower()

        # Extract relevant context
        context = ""
        if query:
            lines = doc_text.split("\n")
            for i, line in enumerate(lines):
                if query in line.lower():
                    start = max(0, i - 10)
                    end = min(len(lines), i + 500)
                    context = "\n".join(lines[start:end])
                    break

        # Use filtered context or fallback
        base_context = context if context else doc_text[:5000]

        prompt = f"""Use the provided context to answer the question.

Context:
{base_context}

Question: {item['question']}
Answer:"""

        gen_data = generate(model, tokenizer, prompt)
        prediction = gen_data["text"]

        # Retry with full document if weak answer
        if ("not found" in prediction.lower() or
            "not mentioned" in prediction.lower() or
            len(prediction) < 10):

            print(f"Fallback for {item['id']}")

            full_prompt = f"""Answer using the FULL document.

Document:
{doc_text}

Question: {item['question']}
Answer:"""

            gen_data = generate(model, tokenizer, full_prompt)
            prediction = gen_data["text"]

        result = {
            "qa_id": item["id"],
            "system": "Context-Filtered",
            "latency": gen_data["latency"],
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"])
        }

        answer = {
            "qa_id": item["id"],
            "question": item["question"],
            "prediction": prediction,
            "gold_answer": item["answer_text"]
        }
        torch.cuda.empty_cache()
        return result, answer
        

    outputs = [process(item) for item in qa_data]

    for r, a in outputs:
        results.append(r)
        answers.append(a)

    with open("answers_context_g.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

Run

In [13]:
docs = load_documents()
qa_data = load_all_qa()

# # Run for Phi
# tok_m, model_m = load_model(PHI)

#index, chunks = build_index(docs, tok_m)

results = []
# results += run_prompt_only(model_m, tok_m, docs, qa_data)
# results += run_rag(model_m, tok_m, index, chunks, qa_data)
# results += run_context(model_m, tok_m, docs, qa_data)

# Repeat for Qwen
tok_q, model_q = load_model(QWEN)

# index, chunks = build_index(docs, tok_q)

# results += run_prompt_only(model_q, tok_q, docs, qa_data)
# results += run_rag(model_q, tok_q, index, chunks, qa_data)
# results += run_context(model_q, tok_q, docs, qa_data)

# with open("final_results.json", "w") as f:
#     json.dump(results, f, indent=2)

In [17]:
print("Running Prompt-Only...")
results_prompt_Qwen = run_prompt_only(model_q, tok_q, docs, qa_data)

with open("results_prompt.json", "w") as f:
    json.dump(results_prompt_Qwen, f, indent=2)
print(f"Prompt-Only complete. Saved {len(results_prompt_Qwen)} results.")

Running Prompt-Only...
Prompt-Only complete. Saved 100 results.


In [15]:
print("Building Index and Running RAG...")
index, chunks = build_index(docs, tok_q, embedder)
results_rag_Qwen = run_rag(model_q, tok_q, index, chunks, qa_data)

with open("results_rag.json", "w") as f:
    json.dump(results_rag_Qwen, f, indent=2)
print(f"RAG complete. Saved {len(results_rag_Qwen)} results.")

Building Index and Running RAG...


Batches:   0%|          | 0/246 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RAG complete. Saved 100 results.


In [14]:
print("Running Context-Optimized...")
results_context_Qwen = run_context(docs, qa_data, model_q, tok_q)

with open("results_context.json", "w") as f:
    json.dump(results_context_Qwen, f, indent=2)
print(f"Context-Optimized complete. Saved {len(results_context_Qwen)} results.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running Context-Optimized...
Context-Optimized complete. Saved 100 results.
